# Preprocesamiento para DistilBERT

## 📝 Contexto del Problema
En las fases anteriores, evaluamos modelos de Machine Learning clásico (Regresión Logística, SVM, LightGBM) utilizando técnicas de vectorización tradicionales (TF-IDF / CountVectorizer). Durante la optimización, detectamos un desafío arquitectónico crítico:

1. **Overfitting por Aumentación:** El uso de técnicas de *back-translation* sobre un corpus reducido (700 comentarios originales) provocó que modelos complejos como SVM memorizaran el ruido sintáctico, resultando en caídas severas de rendimiento en el conjunto de Test (overfitting > 20%).
2. **Límites del Enfoque "Bag of Words":** Al eliminar *stopwords* y signos de puntuación, destruimos el contexto semántico (ej. gritos en mayúsculas, negaciones condicionales), limitando nuestro **Recall** máximo.

## 🎯 Objetivo de este Notebook
Este notebook marca un cambio de paradigma hacia el **Deep Learning**. Vamos a preparar los datos para realizar *Fine-Tuning* sobre un modelo Transformer pre-entrenado (`distilbert-base-uncased` de Hugging Face). 

Al utilizar *Transfer Learning*, aprovecharemos el conocimiento gramatical y semántico que el modelo ya posee del idioma inglés, permitiéndonos entrenar de forma robusta utilizando **únicamente los 1,000 comentarios originales** de nuestro dataset.

## ⚙️ Estrategia de Preprocesamiento (Minimalista)
A diferencia de los modelos clásicos donde utilizábamos `spaCy` o `NLTK` para reducir la dimensionalidad (lematización, eliminación de stopwords), los Transformers necesitan leer el texto tal y como lo escribe un humano.

**Nuestro pipeline aplicará una limpieza estrictamente quirúrgica:**
* ✅ **SE ELIMINA:** URLs, menciones de usuarios (`@user`) y espacios múltiples (basura técnica que no aporta valor semántico).
* ❌ **SE CONSERVA:** Mayúsculas, signos de puntuación (`!!!`, `???`), stopwords y la raíz original de las palabras (esencial para detectar el tono y la agresividad real del usuario).
* 📏 **SE FILTRA:** Se descartarán comentarios algorítmicamente inútiles (menores a 3 palabras o mayores a 200).

## 🚀 Flujo de Trabajo
1. Carga del dataset *raw* original sin aumentación.
2. Aplicación de funciones de limpieza mediante Expresiones Regulares (Regex).
3. Filtrado de *outliers* de longitud.
4. Estandarización de columnas al formato requerido por la librería `datasets` de Hugging Face (`text` y `label`).
5. Exportación del set de datos preparado para su ingesta en Google Colab (entrenamiento en GPU).

In [1]:
import pandas as pd
import re

print("="*80)
print(" PREPROCESAMIENTO MINIMALISTA (VERSIÓN APP LIGERA)")
print("="*80)

# 1. Cargar el dataset original exacto
ruta_dataset = '../../data/raw/youtoxic_original.csv' 
df = pd.read_csv(ruta_dataset)

print(f"✅ Dataset original cargado: {df.shape[0]} filas.")

# 2. Función de limpieza quirúrgica (Regex)
def preprocesar_para_bert(texto):
    texto = str(texto)
    # Quitar URLs
    texto = re.sub(r'http\S+|www.\S+', '', texto)
    # Quitar menciones
    texto = re.sub(r'@\w+', '', texto)
    # Quitar espacios extra
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

# 3. Aplicar limpieza
print("🧹 Aplicando limpieza minimalista...")
df['Text_Clean'] = df['Text'].apply(preprocesar_para_bert)

# 4. Filtrar basura extrema por longitud
df['word_count'] = df['Text_Clean'].apply(lambda x: len(x.split()))
df_filtrado = df[(df['word_count'] >= 3) & (df['word_count'] <= 200)].copy()
print(f"✅ Filas útiles tras filtrar: {df_filtrado.shape[0]}")

# =====================================================================
#  EXPORTACIÓN 1: MASTER PARA LA APLICACIÓN Y BASE DE DATOS
# =====================================================================
# Seleccionamos EXCLUSIVAMENTE las columnas que necesita tu sistema
columnas_app = ['CommentId', 'VideoId', 'Text', 'Text_Clean', 'IsToxic']
df_master = df_filtrado[columnas_app].copy()

nombre_master = '../../data/processed_rn/youtoxic_rn_limpio.csv'
df_master.to_csv(nombre_master, index=False)
print(f"\n📦 1. MASTER GUARDADO: '{nombre_master}'")
print(f"   -> Contiene SOLO: {', '.join(columnas_app)}")

# =====================================================================
#  EXPORTACIÓN 2: DATASET ESTRICTO PARA GOOGLE COLAB (HUGGING FACE)
# =====================================================================
# Nos quedamos SOLO con Text_Clean y IsToxic
df_hf = df_filtrado[['Text_Clean', 'IsToxic']].copy()
df_hf.columns = ['text', 'label'] 
# Asegurarnos de que el target sea 0 y 1 para la Red Neuronal
df_hf['label'] = df_hf['label'].astype(int)

nombre_hf = '../../data/processed_rn/dataset_rn_entrenamiento.csv'
df_hf.to_csv(nombre_hf, index=False)
print(f" 2. DATASET DE ENTRENAMIENTO GUARDADO: '{nombre_hf}'")
print("   -> Solo contiene 'text' y 'label'. Sube ESTE archivo a Google Colab.")
print("="*80)

 PREPROCESAMIENTO MINIMALISTA (VERSIÓN APP LIGERA)
✅ Dataset original cargado: 1000 filas.
🧹 Aplicando limpieza minimalista...
✅ Filas útiles tras filtrar: 953

📦 1. MASTER GUARDADO: '../../data/processed_rn/youtoxic_rn_limpio.csv'
   -> Contiene SOLO: CommentId, VideoId, Text, Text_Clean, IsToxic
 2. DATASET DE ENTRENAMIENTO GUARDADO: '../../data/processed_rn/dataset_rn_entrenamiento.csv'
   -> Solo contiene 'text' y 'label'. Sube ESTE archivo a Google Colab.
